# Pydantic AI Lab Exercise

Solution to the exercise at the end of Week 5 Day 2 (`5_agent_frameworks/2_strands_pydantic/pydantic_lab.ipynb`). The exercise has two parts:

1. Seed a different goal on the board, for example a short haiku about Madrid written to `madrid.txt`, and run the worker again. Does it plan sensible steps and pick the right file tools?
2. After the run, inspect `result.all_messages()` to see the full transcript: every tool call the loop made and every result it read back.

Run the cells top to bottom with the repo's Python 3.12 kernel. The notebook is self-contained: it keeps its own board file and its own `workspace` folder in this directory, so the lab's board and workspace are untouched.

## Setup

`board.py` lives in the day 2 folder, so instead of copying it we put that folder on `sys.path` and import it from there. `BOARD_PATH` must be set before the import: it points the board at a local `board.sqlite` in this folder, which is what keeps our runs off the lab's board.

The model is the same provider-and-model string as the lab, `openai-chat:gpt-5.4-mini`, where the `-chat` pins OpenAI's Chat Completions API explicitly.

In [ ]:
import os
import sys
from pathlib import Path

# This notebook lives four levels below 5_agent_frameworks, so the day 2 folder is here:
DAY2_FOLDER = Path("../../../../2_strands_pydantic").resolve()
sys.path.insert(0, str(DAY2_FOLDER))

os.environ["BOARD_PATH"] = str(Path("board.sqlite").resolve())  # our own board, not the lab's

from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

import board

load_dotenv(override=True)

MODEL = "openai-chat:gpt-5.4-mini"

## The board tools, unchanged from the lab

The three board tools are copied verbatim: `show_todos` reads the board, `plan_steps` breaks a goal into steps, `complete_task` ticks one off. In Pydantic AI they are plain typed functions with no decorator at all; Pydantic reads the type hints and the docstring and builds the JSON schema for the model.

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

## The filesystem MCP server

The same reference server as the lab, started over `npx` and scoped to a `workspace` folder inside this directory, so the agent can only touch files in there. In Pydantic AI the server is a toolset: it goes in `toolsets=[...]`, separate from the plain `tools=[...]` list, and its connection is opened with `async with agent:` around the run.

`log_file=Path(os.devnull)` keeps the server's startup logging off the screen and is also what lets it run from a Jupyter kernel on Windows.

In [ ]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(exist_ok=True)

filesystem = MCPToolset(
    StdioTransport(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
        cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        log_file=Path(os.devnull),
    ),
    init_timeout=60
)

## Task 1: a different goal

Seed the haiku goal and let the worker run. The worker and its instruction are the same as the lab's; only the goal on the board is new.

Unlike Strands, Pydantic AI does not stream the loop to the screen by default, so the run is quiet until the final output. The board afterwards, and the transcript in task 2, show everything that happened in between.

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    MODEL,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task],
    toolsets=[filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with worker:
    result = await worker.run("Please work the pending goal on the board.")
print(result.output)

Now check the outcome: the board should show the goal and its steps struck through, and `madrid.txt` should hold the haiku.

In [ ]:
board.show_board()
print("\nmadrid.txt:\n" + (workspace / "madrid.txt").read_text(encoding="utf-8"))

## Task 2: inspect the transcript with result.all_messages()

The result object carries the whole conversation the loop had: `result.all_messages()` returns the model requests and responses in order. Each message is made of typed parts, and the `part_kind` field says what each part is: the system prompt, the user prompt, a text reply, a `tool-call` the model decided to make, or a `tool-return` carrying what the tool sent back.

The helper below prints one line per part, truncated so the shape of the loop stays visible: decide, call, read, decide again, until the final text.

In [ ]:
def show_transcript(messages) -> None:
    """Print one truncated line per message part, so the loop's shape is easy to scan."""
    for message in messages:
        for part in message.parts:
            kind = part.part_kind
            if kind == "tool-call":
                detail = f"{part.tool_name}({part.args})"
            elif kind == "tool-return":
                detail = f"{part.tool_name} -> {part.content}"
            elif kind == "text":
                detail = part.content
            elif kind == "user-prompt":
                detail = str(part.content)
            else:
                detail = ""   # the system prompt; its text is the INSTRUCTIONS above
            print(f"{kind:13} {str(detail)[:140]}")

show_transcript(result.all_messages())